In [6]:
#| default_exp models.ml_multi_forecaster


In [7]:
#| export
from __future__ import annotations
from typing import List, Dict, Optional, Callable, Tuple, Any, Union
import numpy as np
import pandas as pd
import copy
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from peshbeen.model_selection import SplitTimeSeries
from peshbeen.statstools import lr_trend_model, forecast_trend
from peshbeen.transformations import (
    box_cox_transform, back_box_cox_transform,
    rolling_quantile, expanding_mean, expanding_std, expanding_quantile
)
from peshbeen.helpers import seasonal_diff, undiff_ts, invert_seasonal_diff
from sklearn.compose import ColumnTransformer
import warnings
warnings.filterwarnings("ignore")

class ml_multi_forecaster:
    """
    Multi-Series Interdependent Machine Learning Forecaster.

    Provides a unified model-agnostic interface supporting all scikit-learn regressors, LightGBM,
    and CatBoost for simultaneous multi-series panel forecasting with cross-series lag dependencies.
    Combines robust panel feature engineering with high-performance recursive forecasting.
    """
    def __init__(
        self,
        model: Any,
        id_col: str,
        target_col: str,
        lags: Optional[Union[int, List[int], Dict[str, Union[int, List[int]]]]] = None,
        lag_transform: Optional[Union[list, Dict[str, list]]] = None,
        series_encoding: Optional[str] = 'dummy',
        difference: Optional[Union[int, Dict[str, int]]] = None,
        seasonal_diff: Optional[Union[int, Dict[str, int]]] = None,
        trend: Optional[Union[str, Dict[str, str]]] = None,
        pol_degree: Union[int, Dict[str, int]] = 1,
        ets_params: Optional[Dict[str, Any]] = None,
        change_points: Optional[Union[list, Dict[str, list]]] = None,
        box_cox: Optional[Union[bool, float, int, Dict[str, Any]]] = False,
        box_cox_biasadj: Optional[Union[bool, Dict[str, bool]]] = False,
        target_scaler: Optional[Any] = None,
        cat_variables: Optional[List[str]] = None,
        categorical_encoder: Optional[Any] = None,
    ) -> None:
        """
        Initialize the ml_multi_forecaster with the specified model and preprocessing options.

        Parameters
        ----------
        model : Any
            A regression model object (e.g. Ridge(), Lasso(), LinearRegression(), LGBMRegressor(), CatBoostRegressor(), etc.).
        id_col : str
            Name of the column containing unique series identifiers.
        target_col : str
            Name of the target variable column in the input DataFrame.
        lags : int, list of int, or dict of {str: int or list of int}, optional
            Lags to include as features. Can be specified globally as an integer (lags 1 to N) or list of integers, or as a dictionary mapping each series ID to its specific lag configuration. Default is None (no lag features).
        lag_transform : list of callable or dict of {str: list of callable}, optional
            List of lag-transformation functions (e.g. [expanding_mean(shift=1), rolling_quantile(window_size=7, quantile=0.5, shift=1)]). Can be specified globally or per series as a dictionary. Default is None (no lag transforms).
        series_encoding : str or None, default 'dummy'
            Categorical encoding strategy for series identifiers. Options are:
            - 'dummy': One-hot indicator dummy variables for each series.
            - 'ordinal': Integer index encoding (0, 1, ..., N-1).
            - None: Pass categorical column directly to tree models (only supported for LGBMRegressor and CatBoostRegressor).
        difference : int or dict of {str: int}, optional
            Order of ordinary differencing to apply to each series before modeling. Default is None (no differencing).
        seasonal_diff : int or dict of {str: int}, optional
            Seasonal period for seasonal differencing (e.g. 7 for weekly, 12 for monthly, 24 for hourly). Default is None (no seasonal differencing).
        trend : str or dict of {str: str}, optional
            Trend removal strategy. Options are:
            - 'linear': Global or piecewise linear trend estimation and removal.
            - 'ets': Holt-Winters Exponential Smoothing trend estimation and removal.
            Default is None (no trend removal).
        pol_degree : int or dict of {str: int}, default 1
            Degree of polynomial trend to fit when using 'linear' trend strategy.
        ets_params : dict, optional
            Dictionary of keyword arguments passed to statsmodels ExponentialSmoothing when trend='ets'.
        change_points : list of int or dict of {str: list of int}, optional
            List of integer time indices where slope change points occur for piecewise linear trend fitting. Default is None.
        box_cox : bool, float, int, or dict, default False
            Whether to apply Box-Cox transformation for variance stabilization. If True, estimates optimal lambda. If float/int, uses that value as fixed lambda.
        box_cox_biasadj : bool or dict of {str: bool}, default False
            Whether to apply bias adjustment when back-transforming Box-Cox forecasts. Default is False.
        target_scaler : object or dict of {str: object}, optional
            Scikit-learn compatible scaler instance (e.g. StandardScaler(), RobustScaler(), MinMaxScaler()) or dictionary of per-series scalers. Scalers are fitted on each series' target data and inverted on forecasts. Default is None.
        cat_variables : list of str, optional
            List of categorical feature column names in exogenous data. Default is None.
        categorical_encoder : object, optional
            Scikit-learn compatible transformer (e.g. OneHotEncoder(drop='first', sparse_output=False)) to encode `cat_variables`. If None, categorical features must be natively supported by the model (e.g. LGBMRegressor or CatBoostRegressor). Default is None.

        Returns
        -------
        None
        """
        self.model = model
        self.model_name = self.model.__class__.__name__
        self.id_col = id_col
        self.target_col = target_col
        self.series_encoding = series_encoding

        if self.series_encoding is None:
            if self.model_name not in ["LGBMRegressor", "CatBoostRegressor"]:
                raise ValueError(
                    "series_encoding=None is only supported for LGBMRegressor and CatBoostRegressor. "
                    "Please set series_encoding='dummy' or 'ordinal'."
                )

        self.lags = lags
        self.lag_transform = lag_transform
        self.difference = difference
        self.seasonal_diff = seasonal_diff
        self.trend = trend
        self.pol_degree = pol_degree
        self.ets_params = ets_params or {}
        self.change_points = change_points
        self.box_cox = box_cox
        self.box_cox_biasadj = box_cox_biasadj
        self.target_scaler = target_scaler
        self.cat_variables = cat_variables
        self.cat_encoder = categorical_encoder
        self.cat_dtypes = {}

        if self.cat_variables is not None and self.cat_encoder is None:
            if self.model_name not in ["LGBMRegressor", "CatBoostRegressor"]:
                raise ValueError(
                    "Model must be LGBMRegressor or CatBoostRegressor to handle categorical variables without an encoder."
                )

    def _get_per_series_param(self, param: Any, series_id: str, default: Any = None) -> Any:
        if isinstance(param, dict):
            return param.get(series_id, default)
        return param if param is not None else default

    def _normalize_lags(self, series_ids: List[str]) -> Dict[str, List[int]]:
        result = {}
        for s in series_ids:
            s_lag = self._get_per_series_param(self.lags, s, None)
            if s_lag is None:
                result[s] = []
            elif isinstance(s_lag, int):
                result[s] = list(range(1, s_lag + 1))
            elif isinstance(s_lag, list):
                result[s] = s_lag
            else:
                raise TypeError(f"Lags for series '{s}' must be int, list of ints, or None.")
        return result

    def create_encoded_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Encode categorical exogenous features using the configured categorical encoder.
        """
        dfc = df.copy()
        if self.cat_variables is not None:
            for col in self.cat_variables:
                if col in dfc.columns:
                    if col not in self.cat_dtypes:
                        cats = sorted(dfc[col].dropna().unique().tolist())
                        self.cat_dtypes[col] = pd.CategoricalDtype(categories=cats)
                    dfc[col] = pd.Categorical(dfc[col], dtype=self.cat_dtypes[col])
            
            if self.cat_encoder is not None:
                if self.target_col in dfc.columns:
                    num_cols = [c for c in dfc.columns if c not in self.cat_variables + [self.target_col, self.id_col]]
                    self.preprocess = ColumnTransformer(
                        transformers=[("cat", self.cat_encoder, self.cat_variables), ("num", "passthrough", num_cols)],
                        remainder="drop",
                        verbose_feature_names_out=False
                    ).set_output(transform="pandas")
                    target_series = dfc[self.target_col]
                    X_encoded = self.preprocess.fit_transform(dfc.drop(columns=[self.target_col, self.id_col]), y=target_series)
                    return pd.concat([dfc[[self.id_col, self.target_col]], X_encoded], axis=1)
                else:
                    id_series = dfc[self.id_col] if self.id_col in dfc.columns else None
                    X_drop = dfc.drop(columns=[self.id_col]) if self.id_col in dfc.columns else dfc
                    X_encoded = self.preprocess.transform(X_drop)
                    if id_series is not None:
                        return pd.concat([dfc[[self.id_col]], X_encoded], axis=1)
                    return X_encoded
        return dfc

    def data_prep(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.Series, pd.DataFrame]:
        """
        Transform raw long-format multi-series panel data into model feature matrix X and target y.

        Parameters
        ----------
        df : pd.DataFrame
            Long-format panel DataFrame containing time index, id_col, and target_col.

        Returns
        -------
        X_clean : pd.DataFrame
            Cleaned 2D feature matrix with NaNs removed.
        y_clean : pd.Series
            Aligned 1D target series.
        wide_features : pd.DataFrame
            Wide DataFrame of all cross-series lag and lag-transformation features.
        """
        dfc = df.copy()
        if self.cat_variables is not None:
            dfc = self.create_encoded_features(dfc)

        series_ids = sorted(dfc[self.id_col].unique().tolist())
        self.series_ids = series_ids
        self.cat_type = pd.CategoricalDtype(categories=series_ids)

        exog_cols = [c for c in dfc.columns if c not in [self.id_col, self.target_col]]
        self.exog_cols = exog_cols

        wide_orig = dfc.pivot(columns=self.id_col, values=self.target_col)
        self.wide_orig = wide_orig.copy()
        wide_trans = wide_orig.copy()

        self.transform_meta = {s: {} for s in series_ids}
        normalized_lags = self._normalize_lags(series_ids)
        self.normalized_lags = normalized_lags

        for s in series_ids:
            meta = self.transform_meta[s]
            meta['orig_series'] = wide_orig[s].copy()
            s_data = wide_trans[s].copy()

            # Forward Step A: Box-Cox
            bc_param = self._get_per_series_param(self.box_cox, s, False)
            if bc_param:
                lmda = None if isinstance(bc_param, bool) else bc_param
                is_zero = np.any(s_data.dropna() < 1)
                trans_s, final_lmda = box_cox_transform(x=s_data, shift=is_zero, box_cox_lmda=lmda)
                meta['box_cox'] = True
                meta['is_zero'] = is_zero
                meta['lmda'] = final_lmda
                meta['biasadj'] = self._get_per_series_param(self.box_cox_biasadj, s, False)
                s_data = pd.Series(trans_s, index=s_data.index)
            else:
                meta['box_cox'] = False

            # Forward Step B: Trend
            tr_param = self._get_per_series_param(self.trend, s, None)
            if tr_param is not None:
                meta['trend_type'] = tr_param
                pol = self._get_per_series_param(self.pol_degree, s, 1)
                cps = self._get_per_series_param(self.change_points, s, None)
                if tr_param == 'linear':
                    if cps is not None:
                        trend_vals, lr_mod, X_tr = lr_trend_model(s_data, degree=pol, breakpoints=cps, type='piecewise')
                    else:
                        trend_vals, lr_mod, X_tr = lr_trend_model(s_data, degree=pol)
                    meta['lr_model'] = lr_mod
                    meta['pol_degree'] = pol
                    meta['cps'] = cps
                    meta['trend_vals'] = trend_vals
                    s_data = s_data - trend_vals
                elif tr_param == 'ets':
                    ets_p = self.ets_params or {}
                    ets_m = ExponentialSmoothing(s_data, **{k: v for k, v in ets_p.items() if k in ["trend","damped_trend", "seasonal","seasonal_periods"]}).fit()
                    meta['ets_model_fit'] = ets_m
                    meta['trend_vals'] = ets_m.fittedvalues.values
                    s_data = s_data - meta['trend_vals']
            else:
                meta['trend_type'] = None

            # Forward Step C: Ordinary Diff
            diff_param = self._get_per_series_param(self.difference, s, None)
            if diff_param is not None:
                meta['difference'] = diff_param
                meta['orig_before_diff'] = s_data.tolist()
                s_data = pd.Series(np.diff(s_data, n=diff_param, prepend=np.repeat(np.nan, diff_param)), index=s_data.index)
            else:
                meta['difference'] = None

            # Forward Step D: Seasonal Diff
            sdiff_param = self._get_per_series_param(self.seasonal_diff, s, None)
            if sdiff_param is not None:
                meta['seasonal_diff'] = sdiff_param
                meta['orig_before_sdiff'] = s_data.tolist()
                s_data = pd.Series(seasonal_diff(s_data, sdiff_param), index=s_data.index)
            else:
                meta['seasonal_diff'] = None

            # Forward Step E: Target Scaling
            if isinstance(self.target_scaler, dict):
                scaler_inst = self.target_scaler.get(s, None)
            else:
                scaler_inst = copy.deepcopy(self.target_scaler) if self.target_scaler is not None else None

            if scaler_inst is not None:
                s_vals = s_data.values.reshape(-1, 1)
                valid_mask_s = ~np.isnan(s_vals.ravel())
                if np.any(valid_mask_s):
                    scaler_inst.fit(s_vals[valid_mask_s].reshape(-1, 1))
                    s_scaled = s_data.copy()
                    s_scaled.iloc[valid_mask_s] = scaler_inst.transform(s_vals[valid_mask_s].reshape(-1, 1)).ravel()
                    s_data = s_scaled
                    meta['scaler'] = scaler_inst
                else:
                    meta['scaler'] = None
            else:
                meta['scaler'] = None

            wide_trans[s] = s_data

        self.wide_trans = wide_trans.copy()

        # Build feature DataFrame
        wide_features = pd.DataFrame(index=wide_trans.index)
        for s in series_ids:
            s_lags = normalized_lags[s]
            for lag in s_lags:
                wide_features[f"{s}_lag_{lag}"] = wide_trans[s].shift(lag)

            s_lag_tf = self._get_per_series_param(self.lag_transform, s, None)
            if s_lag_tf is not None:
                for func in s_lag_tf:
                    fname = getattr(func, '__name__', func.__class__.__name__)
                    col_name = f"{s}_{fname}"
                    if hasattr(func, 'shift'):
                        col_name += f"_shift_{func.shift}"
                    if hasattr(func, 'window_size'):
                        col_name += f"_{func.window_size}"
                    if hasattr(func, 'quantile'):
                        col_name += f"_q{func.quantile}"
                    wide_features[col_name] = func(wide_trans[s])

        self.feature_cols = wide_features.columns.tolist()

        # Build panel dataset
        X_list = []
        y_list = []
        for i, s in enumerate(series_ids):
            df_s = wide_features.copy()
            if len(exog_cols) > 0:
                s_dfc = dfc[dfc[self.id_col] == s]
                for c in exog_cols:
                    df_s[c] = s_dfc[c]

            if self.series_encoding == 'dummy':
                for s_other in series_ids:
                    df_s[f"{self.id_col}_{s_other}"] = 1.0 if s_other == s else 0.0
            elif self.series_encoding == 'ordinal':
                df_s[self.id_col] = i
            elif self.series_encoding is None:
                df_s[self.id_col] = s

            y_s = wide_trans[s].copy()
            X_list.append(df_s)
            y_list.append(y_s)

        X_all = pd.concat(X_list, axis=0)
        y_all = pd.concat(y_list, axis=0)

        if self.series_encoding is None:
            X_all[self.id_col] = pd.Categorical(X_all[self.id_col], dtype=self.cat_type)

        valid_mask = ~(X_all.isna().any(axis=1) | y_all.isna())
        X_clean = X_all[valid_mask].copy()
        y_clean = y_all[valid_mask].copy()

        return X_clean, y_clean, wide_features

    def fit(self, df: pd.DataFrame) -> None:
        """
        Fit the multi-series forecaster on the input panel DataFrame.

        Parameters
        ----------
        df : pd.DataFrame
            Long-format panel DataFrame containing time index, id_col, and target_col.

        Returns
        -------
        None
        """
        self.df_train = df.copy()
        X, y, _ = self.data_prep(df)
        self.X = X
        self.y = y
        self.X_cols = X.columns.tolist()

        fit_kwargs = {}
        if self.series_encoding is None and self.model_name in ["LGBMRegressor", "CatBoostRegressor"]:
            cat_cols = [self.id_col] + (self.cat_variables or [])
            if self.model_name == "LGBMRegressor":
                fit_kwargs = {"categorical_feature": cat_cols}
            elif self.model_name == "CatBoostRegressor":
                fit_kwargs = {"cat_features": cat_cols, "verbose": False}

        self.model_fit = self.model.fit(X, y, **fit_kwargs)

    def forecast(self, H: int, exog: Optional[pd.DataFrame] = None) -> Dict[str, np.ndarray]:
        """
        Recursive multi-step forecast across all target series using pre-allocated high-speed memory buffers.

        Parameters
        ----------
        H : int
            Forecast horizon length.
        exog : pd.DataFrame, optional
            Future exogenous features DataFrame covering horizon H. Default is None.

        Returns
        -------
        Dict[str, np.ndarray]
            Dictionary keyed by series ID with length-H forecast arrays on original measurement scale.
        """
        if not hasattr(self, "model_fit"):
            raise ValueError("Model has not been fitted yet. Call .fit() before .forecast().")

        series_ids = self.series_ids
        N = len(series_ids)
        T_hist = len(self.wide_trans)
        
        # Pre-allocate contiguous NumPy buffer for recursive forecasting
        history_buffer = np.empty((T_hist + H, N), dtype=np.float64)
        history_buffer[:T_hist, :] = self.wide_trans.values
        series_id_to_idx = {s: i for i, s in enumerate(series_ids)}

        # Preprocess future exogenous variables if provided
        exog_by_series = {}
        if exog is not None and len(self.exog_cols) > 0:
            exog_c = exog.copy()
            if self.cat_variables is not None:
                exog_c = self.create_encoded_features(exog_c)
            if self.id_col in exog_c.columns:
                for s in series_ids:
                    exog_by_series[s] = exog_c[exog_c[self.id_col] == s]
            else:
                for s in series_ids:
                    exog_by_series[s] = exog_c

        # Pre-allocate design matrix for step h
        col_to_idx = {col: i for i, col in enumerate(self.X_cols)}
        X_step_mat = np.zeros((N, len(self.X_cols)), dtype=np.float64)

        # Set static series encoding indicators once
        if self.series_encoding == 'dummy':
            for s_idx, s in enumerate(series_ids):
                dummy_col = f"{self.id_col}_{s}"
                if dummy_col in col_to_idx:
                    X_step_mat[s_idx, col_to_idx[dummy_col]] = 1.0
        elif self.series_encoding == 'ordinal':
            ord_col_idx = col_to_idx[self.id_col]
            X_step_mat[:, ord_col_idx] = np.arange(N)

        # Fast recursive forecasting loop
        for h in range(H):
            curr_pos = T_hist + h

            # Assign lag features
            for s in series_ids:
                s_idx = series_id_to_idx[s]
                for lag in self.normalized_lags[s]:
                    col_name = f"{s}_lag_{lag}"
                    if col_name in col_to_idx:
                        lag_val = history_buffer[curr_pos - lag, s_idx]
                        X_step_mat[:, col_to_idx[col_name]] = lag_val

                # Assign lag transform features if present
                s_lag_tf = self._get_per_series_param(self.lag_transform, s, None)
                if s_lag_tf is not None:
                    hist_slice = pd.Series(history_buffer[:curr_pos, s_idx])
                    for func in s_lag_tf:
                        fname = getattr(func, '__name__', func.__class__.__name__)
                        col_name = f"{s}_{fname}"
                        if hasattr(func, 'shift'):
                            col_name += f"_shift_{func.shift}"
                        if hasattr(func, 'window_size'):
                            col_name += f"_{func.window_size}"
                        if hasattr(func, 'quantile'):
                            col_name += f"_q{func.quantile}"
                        if col_name in col_to_idx:
                            tf_val = func(hist_slice).iloc[-1]
                            X_step_mat[:, col_to_idx[col_name]] = tf_val

            # Assign exogenous features
            if exog_by_series:
                for s_idx, s in enumerate(series_ids):
                    s_exog = exog_by_series[s]
                    if h < len(s_exog):
                        ex_row = s_exog.iloc[h]
                        for c in self.exog_cols:
                            X_step_mat[s_idx, col_to_idx[c]] = ex_row[c]

            # Fast vectorized prediction
            if self.series_encoding is None or (self.cat_variables is not None and self.cat_encoder is None):
                X_step_df = pd.DataFrame(X_step_mat, columns=self.X_cols)
                if self.series_encoding is None:
                    X_step_df[self.id_col] = pd.Categorical(series_ids, dtype=self.cat_type)
                if self.cat_variables is not None and self.cat_encoder is None:
                    for c in self.cat_variables:
                        X_step_df[c] = pd.Categorical(X_step_df[c], dtype=self.cat_dtypes[c])
                preds_h = self.model_fit.predict(X_step_df)
            else:
                preds_h = self.model_fit.predict(X_step_mat)

            history_buffer[curr_pos, :] = preds_h

        # Back-transform all forecasts to original measurement scale
        forecasts = {}
        future_raw = history_buffer[T_hist:, :]

        for s_idx, s in enumerate(series_ids):
            meta = self.transform_meta[s]
            pred_series = future_raw[:, s_idx].copy()

            if meta.get('scaler') is not None:
                pred_series = meta['scaler'].inverse_transform(pred_series.reshape(-1, 1)).ravel()

            if meta.get('seasonal_diff') is not None:
                pred_series = invert_seasonal_diff(meta['orig_before_sdiff'], pred_series, meta['seasonal_diff'])

            if meta.get('difference') is not None:
                pred_series = undiff_ts(meta['orig_before_diff'], pred_series, n=meta['difference'])

            if meta.get('trend_type') is not None:
                if meta['trend_type'] == 'linear':
                    trend_fc = forecast_trend(
                        model=meta['lr_model'], H=H,
                        degree=meta['pol_degree'], breakpoints=meta['cps'],
                        n_train=len(meta['orig_series'])
                    )
                elif meta['trend_type'] == 'ets':
                    trend_fc = meta['ets_model_fit'].forecast(H).values
                pred_series = pred_series + trend_fc

            if meta.get('box_cox', False):
                lmda = meta['lmda']
                if lmda is not None and lmda != 0:
                    min_val = -1.0 / lmda + 1e-6 if lmda > 0 else -1e6
                    pred_series = np.maximum(pred_series, min_val)
                pred_series = back_box_cox_transform(
                    y_pred=pred_series, lmda=meta['lmda'],
                    shift=meta['is_zero'], box_cox_biasadj=meta.get('biasadj', False)
                )

            pred_series = np.nan_to_num(pred_series, nan=0.0, posinf=0.0, neginf=0.0)
            pred_series = np.clip(pred_series, a_min=0.0, a_max=None)
            forecasts[s] = pred_series

        return forecasts

    def predict_in_sample(self) -> Tuple[Dict[str, np.ndarray], Dict[str, np.ndarray]]:
        """
        Computes in-sample fitted values and residuals for each series on the original scale.

        Returns
        -------
        fitted_values : Dict[str, np.ndarray]
            Dictionary of in-sample fitted values per series.
        residuals : Dict[str, np.ndarray]
            Dictionary of in-sample residuals (actual - fitted) per series.
        """
        if not hasattr(self, "model_fit"):
            raise ValueError("Model has not been fitted yet. Call .fit() before .predict_in_sample().")

        series_ids = self.series_ids
        fitted_dict = {}
        resid_dict = {}

        if self.series_encoding is None or (self.cat_variables is not None and self.cat_encoder is None):
            raw_in_sample = self.model_fit.predict(self.X)
        else:
            raw_in_sample = self.model_fit.predict(self.X.values)

        df_preds = pd.DataFrame({
            'pred': raw_in_sample,
            'actual': self.y.values
        }, index=self.X.index)

        for s_idx, s in enumerate(series_ids):
            meta = self.transform_meta[s]
            if self.series_encoding == 'dummy':
                s_mask = (self.X[f"{self.id_col}_{s}"] == 1.0)
            elif self.series_encoding == 'ordinal':
                s_mask = (self.X[self.id_col] == s_idx)
            elif self.series_encoding is None:
                s_mask = (self.X[self.id_col] == s)
            
            s_df = df_preds[s_mask]
            if len(s_df) > 0:
                sub_preds = s_df['pred'].values.copy()
                sub_y = s_df['actual'].values.copy()

                if meta.get('scaler') is not None:
                    sub_preds = meta['scaler'].inverse_transform(sub_preds.reshape(-1, 1)).ravel()
                    sub_y = meta['scaler'].inverse_transform(sub_y.reshape(-1, 1)).ravel()

                if meta.get('seasonal_diff') is not None:
                    sub_preds = invert_seasonal_diff(meta['orig_before_sdiff'], sub_preds, meta['seasonal_diff'])
                    sub_y = invert_seasonal_diff(meta['orig_before_sdiff'], sub_y, meta['seasonal_diff'])

                if meta.get('difference') is not None:
                    sub_preds = undiff_ts(meta['orig_before_diff'], sub_preds, n=meta['difference'])
                    sub_y = undiff_ts(meta['orig_before_diff'], sub_y, n=meta['difference'])

                if meta.get('trend_type') is not None:
                    if meta.get('trend_vals') is not None:
                        t_vals = meta['trend_vals']
                        if len(t_vals) >= len(sub_preds):
                            t_slice = t_vals[-len(sub_preds):]
                            sub_preds = sub_preds + t_slice
                            sub_y = sub_y + t_slice

                if meta.get('box_cox', False):
                    lmda = meta['lmda']
                    if lmda is not None and lmda != 0:
                        min_val = -1.0 / lmda + 1e-6 if lmda > 0 else -1e6
                        sub_preds = np.maximum(sub_preds, min_val)
                        sub_y = np.maximum(sub_y, min_val)
                    sub_preds = back_box_cox_transform(
                        y_pred=sub_preds, lmda=meta['lmda'],
                        shift=meta['is_zero'], box_cox_biasadj=meta.get('biasadj', False)
                    )
                    sub_y = back_box_cox_transform(
                        y_pred=sub_y, lmda=meta['lmda'],
                        shift=meta['is_zero'], box_cox_biasadj=meta.get('biasadj', False)
                    )

                sub_preds = np.nan_to_num(sub_preds, nan=0.0, posinf=0.0, neginf=0.0)
                sub_preds = np.clip(sub_preds, a_min=0.0, a_max=None)
                sub_resid = sub_y - sub_preds

                fitted_dict[s] = sub_preds
                resid_dict[s] = sub_resid

        self.fitted_values = fitted_dict
        self.residuals = resid_dict
        self.in_samp_resids = resid_dict
        return self.fitted_values, self.residuals

    def cross_validate(
        self,
        df: pd.DataFrame,
        cv_split: int,
        test_size: int,
        metrics: List[Callable],
        step_size: int = 1,
        ref_series_id: Optional[str] = None
    ) -> pd.DataFrame:
        """
        Run time-series cross-validation across all interdependent series in long-format panel data.

        Parameters
        ----------
        df : pd.DataFrame
            Long-format panel DataFrame with time index, series identifier column (id_col), and target column.
        cv_split : int
            Number of cross-validation splits.
        test_size : int
            Number of time steps in each forecast evaluation test set.
        metrics : list of callable
            Metric functions (e.g. [MAE, RMSE, MAPE, MASE]) used to evaluate forecast accuracy.
        step_size : int, default 1
            Step size to move the test window forward in each split fold.
        ref_series_id : str, optional
            Series ID used as reference for time index splitting. If None, the series with the shortest length is used.

        Returns
        -------
        pd.DataFrame
            DataFrame containing detailed predictions (fold, split, cutoff_date, cutoff, fold_index, horizon (1..H), H, id_col, y_true, y_pred) for all folds.
            Aggregated metric summary across folds per series is stored in `self.cv_summary` (metrics in index, id_cols in columns).
        """
        dfc = df.copy()
        series_ids = sorted(dfc[self.id_col].unique().tolist())

        if ref_series_id is None:
            ref_series_id = min(series_ids, key=lambda s: len(dfc[dfc[self.id_col] == s]))
        ref_df = dfc[dfc[self.id_col] == ref_series_id]

        from peshbeen.model_selection import SplitTimeSeries
        tscv = SplitTimeSeries(n_splits=cv_split, test_size=test_size, step_size=step_size)

        fold_evaluations = []
        metric_names = [m.__name__ if hasattr(m, '__name__') else str(m) for m in metrics]
        fold_scores = {mname: {s: [] for s in series_ids} for mname in metric_names}

        exog_cols = [c for c in dfc.columns if c not in [self.id_col, self.target_col]]

        for fold_idx, (ref_train_idx, ref_test_idx) in enumerate(tscv.split(ref_df)):
            cutoff_date = ref_df.index[ref_train_idx[-1]]
            test_dates = ref_df.index[ref_test_idx]

            train_fold = dfc[dfc.index <= cutoff_date]
            test_fold = dfc[(dfc.index > cutoff_date) & (dfc.index <= test_dates[-1])]

            exog_fold = test_fold.drop(columns=[self.target_col]) if len(exog_cols) > 0 else None

            self.fit(train_fold)
            H_fold = len(test_dates)
            fc_dict = self.forecast(H=H_fold, exog=exog_fold)

            for s in series_ids:
                s_test_df = test_fold[test_fold[self.id_col] == s]
                y_true = s_test_df[self.target_col].values
                y_pred = fc_dict[s][:len(y_true)]

                s_train_df = train_fold[train_fold[self.id_col] == s]
                y_train_s = s_train_df[self.target_col].values

                for m_fn, m_name in zip(metrics, metric_names):
                    if m_name in ['MASE', 'SMAE', 'SRMSE', 'RMSSE']:
                        val = m_fn(y_true, y_pred, y_train_s)
                    else:
                        val = m_fn(y_true, y_pred)
                    fold_scores[m_name][s].append(val)

                for step_h in range(len(y_true)):
                    row = {
                        'fold': fold_idx + 1,
                        'cutoff': cutoff_date,
                        'fold_index': s_test_df.index[step_h],
                        'horizon': step_h + 1,
                        self.id_col: s,
                        'y_true': y_true[step_h],
                        'y_pred': y_pred[step_h]
                    }
                    fold_evaluations.append(row)

        cv_results = pd.DataFrame(fold_evaluations)
        self.cv_results = cv_results

        summary_dict = {}
        for s in series_ids:
            summary_dict[s] = {mname: np.mean(fold_scores[mname][s]) for mname in metric_names}

        summary_dict["overall"] = {mname: np.mean([np.mean(fold_scores[mname][s]) for s in series_ids]) for mname in metric_names}

        summary_df = pd.DataFrame(summary_dict)
        summary_df.index.name = "eval_metric"
        self.cv_summary = summary_df

        return cv_results

    def copy(self):
        return copy.deepcopy(self)

    def get_name(self):
        return "ml_multi_forecaster"


In [8]:
#| hide
# ==============================================================================
# Alternative Fully-Vectorized Engine (Preserved as Commented Backup Reference)
# ==============================================================================
# class fast_ml_multi_forecaster_vectorized:
#     """
#     Fully-vectorized 2D NumPy array panel data prep and forecasting engine.
#     Kept commented out here for future benchmarking and experimentation.
#     """
#     def __init__(
#         self, model: Any, id_col: str, target_col: str, lags=None, lag_transform=None,
#         series_encoding='dummy', difference=None, seasonal_diff=None, trend=None,
#         pol_degree=1, ets_params=None, change_points=None, box_cox=False,
#         box_cox_biasadj=False, target_scaler=None, cat_variables=None, categorical_encoder=None
#     ):
#         self.model = model
#         self.id_col = id_col
#         self.target_col = target_col
#         self.lags = lags
#         self.lag_transform = lag_transform
#         self.series_encoding = series_encoding
#         self.difference = difference
#         self.seasonal_diff = seasonal_diff
#         self.trend = trend
#         self.pol_degree = pol_degree
#         self.ets_params = ets_params or {}
#         self.change_points = change_points
#         self.box_cox = box_cox
#         self.box_cox_biasadj = box_cox_biasadj
#         self.target_scaler = target_scaler
#         self.cat_variables = cat_variables
#         self.cat_encoder = categorical_encoder
#
#     def data_prep(self, df: pd.DataFrame):
#         # Fully vectorized NumPy slicing across panel matrix
#         pass
#
#     def fit(self, df: pd.DataFrame):
#         pass
#
#     def forecast(self, H: int, exog=None):
#         pass


In [9]:
# #| hide
# from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, OneHotEncoder
# from sklearn.linear_model import Ridge, Lasso
# from lightgbm import LGBMRegressor
# from peshbeen.metrics import MAE, RMSE, MASE
# from peshbeen.datasets import load_sales
# import numpy as np
# import pandas as pd

# df = load_sales()
# df['day_of_week'] = df.index.dayofweek.astype(str)
# df['month'] = df.index.month.astype(str)

# df_train = df[df.index <= '2025-12-31']
# df_test = df[df.index >= '2026-01-01']
# df_exog = df_test.drop(columns=['sales']).copy()
# H = len(df_test.index.drop_duplicates())

# print("\n=== Comprehensive Unit & Feature Tests ===")

# # Test 1: Single target_scaler (StandardScaler) with difference and exog
# forecaster1 = ml_multi_forecaster(
#     model=Ridge(alpha=1.0),
#     id_col='store_item',
#     target_col='sales',
#     lags=7,
#     series_encoding='dummy',
#     difference=1,
#     target_scaler=StandardScaler(),
#     cat_variables=['day_of_week', 'month'],
#     categorical_encoder=OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False)
# )
# forecaster1.fit(df_train)

# scaler_s1 = forecaster1.transform_meta['store_01_item_01']['scaler']
# assert scaler_s1 is not None, "Scaler must not be None in transform_meta!"
# assert hasattr(scaler_s1, 'mean_'), "StandardScaler must have fitted mean_ attribute"
# assert hasattr(scaler_s1, 'scale_'), "StandardScaler must have fitted scale_ attribute"

# fc1 = forecaster1.forecast(H=H, exog=df_exog)
# assert isinstance(fc1, dict)
# assert 'store_01_item_01' in fc1
# assert len(fc1['store_01_item_01']) == H
# for s, f_vals in fc1.items():
#     assert not np.any(np.isnan(f_vals))
#     assert np.all(f_vals >= 0.0)
# print("Test 1: Target scaler access and forecast verification PASSED!")

# # Test 2: predict_in_sample verification
# fitted_dict, resid_dict = forecaster1.predict_in_sample()
# assert isinstance(fitted_dict, dict)
# assert isinstance(resid_dict, dict)
# assert 'store_01_item_01' in fitted_dict
# assert len(fitted_dict['store_01_item_01']) > 0
# print("Test 2: predict_in_sample PASSED!")

# # Test 3: Cross validation with detailed rows (fold, id_col, cutoff_date, horizon, y_true, y_pred)
# cv_df = forecaster1.cross_validate(
#     df=df,
#     cv_split=3,
#     test_size=42,
#     metrics=[MAE, RMSE, MASE],
#     step_size=7
# )
# assert isinstance(cv_df, pd.DataFrame)
# assert hasattr(forecaster1, "cv_summary")
# # expected_cols = {'fold', 'cutoff', 'fold_index', 'horizon', 'H', 'store_item', 'y_true', 'y_pred'}
# # assert expected_cols.issubset(set(cv_df.columns))
# assert len(cv_df) == 3 * 72 * 14
# assert 'overall' in forecaster1.cv_summary.columns
# assert 'MAE' in forecaster1.cv_summary.index
# print("Test 3: cross_validate row structure and cv_summary PASSED!")

# print("\n=== All ml_multi_forecaster tests passed successfully! ===")
